In [29]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn import metrics
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import warnings

warnings.simplefilter(action="ignore")

In [30]:
def load_data(path):
    # read csv data, remove nan rows and rpm
    LABELS = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
    df = pd.read_csv(path, na_values=[" ", ""], skipinitialspace=True)
    df = df.drop(columns=["rpm"], errors="ignore")
    df["bearing_freq_peak_Hz"] = pd.to_numeric(
        df["bearing_freq_peak_Hz"], errors="coerce"
    )
    df["condition_label"] = df["condition_label"].str.strip().str.upper().map(LABELS)
    return df

In [31]:
# Reading and CLeaning data
"""
df = pd.read_csv("train_data.csv", na_values=[" ", ""], skipinitialspace=True)
df = df.dropna()

# the rpm readings were not accurate, and thus won't be considered for training
df = df.drop(columns="rpm")

df["bearing_freq_peak_Hz"] = pd.to_numeric(df["bearing_freq_peak_Hz"], errors="coerce")

# encoding the HIGH, MEDIUM and LOW values
label_map = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
df["condition_label"] = df["condition_label"].str.strip().str.upper().map(label_map)

df.info()

# since this loaded data is considered completely healthy, we shall refer to it as 'healthy'
healthy = df
"""


'\ndf = pd.read_csv("train_data.csv", na_values=[" ", ""], skipinitialspace=True)\ndf = df.dropna()\n\n# the rpm readings were not accurate, and thus won\'t be considered for training\ndf = df.drop(columns="rpm")\n\ndf["bearing_freq_peak_Hz"] = pd.to_numeric(df["bearing_freq_peak_Hz"], errors="coerce")\n\n# encoding the HIGH, MEDIUM and LOW values\nlabel_map = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}\ndf["condition_label"] = df["condition_label"].str.strip().str.upper().map(label_map)\n\ndf.info()\n\n# since this loaded data is considered completely healthy, we shall refer to it as \'healthy\'\nhealthy = df\n'

In [32]:
def decode_labels(codes):
    inverse_map = {0: "LOW", 1: "MEDIUM", 2: "HIGH"}
    return pd.Series(codes).round().astype(int).map(inverse_map)

In [33]:
def unexplained_part(model, scaled):
    """Squash each row down to the learned patterns and rebuild it.
    What doesn't survive the round trip is the 'leftover' (the disagreement)."""
    rebuilt = model["pca"].inverse_transform(model["pca"].transform(scaled))
    return scaled - rebuilt

In [34]:
# PCA model training


def learn(healthy, false_alarm_rate=0.01):
    healthy = healthy.dropna()

    # Put every column on the same scale so none dominates
    scaler = StandardScaler().fit(healthy)
    scaled = scaler.transform(healthy)

    # PCA finds the patterns shared between columns (e.g. temperature moves with ambient)
    pca = PCA(n_components=0.95).fit(scaled)

    # How much of each column the patterns cannot explain (1 = none of it, 0 = all of it)
    unexplained_share = 1 - (pca.components_**2).sum(axis=0)

    model = {
        "columns": list(healthy.columns),
        "scaler": scaler,
        "pca": pca,
        "unexplained_share": unexplained_share,
    }

    # Alert threshold: healthy data should only cross it `false_alarm_rate` of the time
    leftover = unexplained_part(model, scaled)
    disagreement = (leftover**2).sum(axis=1)
    model["threshold"] = np.quantile(disagreement, 1 - false_alarm_rate)
    return model

In [35]:
def test(model, new_data):
    """For every row: is it unusual, which column is the likely culprit,
    how far has that column drifted, and what should its reading have been?"""
    columns = model["columns"]
    share = model["unexplained_share"]
    new_data = new_data[columns]

    report = pd.DataFrame(
        index=new_data.index,
        columns=[
            "disagreement",
            "is_unusual",
            "suspect_column",
            "drift_estimate",
            "corrected_value",
        ],
    )
    report["is_unusual"] = False

    # Rows with empty cells can't be scored
    has_all_values = new_data.notna().all(axis=1)
    report["note"] = np.where(has_all_values, "", "has empty cells - not scored")
    rows = new_data[has_all_values]
    scaled = model["scaler"].transform(rows)

    # How much does each row disagree with the learned patterns?
    leftover = unexplained_part(model, scaled)
    disagreement = (leftover**2).sum(axis=1)
    is_unusual = disagreement > model["threshold"]
    report.loc[rows.index, "disagreement"] = disagreement
    report.loc[rows.index, "is_unusual"] = is_unusual

    # Which single column, if fixed, would remove most of the disagreement?
    blame = leftover**2 / share
    culprit = blame.argmax(axis=1)  # column number for each row
    row_number = np.arange(len(rows))

    # How much it has drifted, converted back to the column's own units
    drift = leftover[row_number, culprit] / share[culprit]
    drift = drift * model["scaler"].scale_[culprit]
    reading = rows.to_numpy()[row_number, culprit]

    # Fill in details only for the unusual rows
    flagged = rows.index[is_unusual]
    report.loc[flagged, "suspect_column"] = np.array(columns)[culprit][is_unusual]
    report.loc[flagged, "drift_estimate"] = drift[is_unusual]
    report.loc[flagged, "corrected_value"] = (reading - drift)[is_unusual]
    return report

In [ ]:
DEFAULT_SENSORS = {
    "temperature_sensor_C": ("true_temp_drift", 0.05),
    "current_sensor_A": ("true_current_drift", 0.005),
    "vibration_rms_g": ("true_vibration_rms_g", 0.05),
}


def compare(report, truth, sensors=DEFAULT_SENSORS):
    """Compare check()'s report with the true drift values.
    Prints a summary and returns (summary, sensor_errors)."""
    if "row" in truth.columns:
        truth = truth.set_index("row")

    flagged = report["is_unusual"].astype(bool)
    scored = report["note"] == ""  # rows without blank cells

    # A row is "truly drifting" if any sensor's true drift is above its tolerance
    truly_drifting = pd.Series(False, index=report.index)
    for sensor, (truth_column, tolerance) in sensors.items():
        truly_drifting |= truth[truth_column].abs() > tolerance

    summary = {
        "caught": int((flagged & truly_drifting).sum()),
        "missed": int((~flagged & truly_drifting & scored).sum()),
        "false_alarms": int((flagged & ~truly_drifting).sum()),
        "drifting_but_not_scored": int((~scored & truly_drifting).sum()),
    }

    # For each sensor: how close were the drift estimates on rows blamed on it?
    sensor_errors = {}
    for sensor, (truth_column, tolerance) in sensors.items():
        blamed = report[flagged & (report["suspect_column"] == sensor)]
        if len(blamed) == 0:
            sensor_errors[sensor] = {"rows_blamed": 0, "mean_abs_error": None}
            continue
        error = (
            blamed["drift_estimate"].astype(float)
            - truth.loc[blamed.index, truth_column]
        )
        sensor_errors[sensor] = {
            "rows_blamed": len(blamed),
            "mean_abs_error": round(error.abs().mean(), 3),
        }
    sensor_errors = pd.DataFrame(sensor_errors).T

    print("Summary:", summary)
    print(sensor_errors)
    return summary, sensor_errors


In [37]:
# since both healthy and test datasets are being loaded,
# the loading and preprocessing has been left to a function
healthy = load_data("train_data.csv")
healthy.info()
healthy

<class 'pandas.DataFrame'>
RangeIndex: 770 entries, 0 to 769
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   condition_label       769 non-null    float64
 1   load_percent          770 non-null    float64
 2   ambient_temp_C        770 non-null    float64
 3   temperature_sensor_C  770 non-null    float64
 4   humidity_percent      770 non-null    float64
 5   current_sensor_A      770 non-null    float64
 6   vibration_rms_g       770 non-null    float64
 7   vibration_peak_g      770 non-null    float64
 8   bearing_freq_peak_Hz  691 non-null    float64
dtypes: float64(9)
memory usage: 54.3 KB


,condition_label,load_percent,ambient_temp_C,temperature_sensor_C,humidity_percent,current_sensor_A,vibration_rms_g,vibration_peak_g,bearing_freq_peak_Hz
0,2.0,0.0,25.3,25.3,80.0,0.001,0.0121,0.0351,NaN
1,2.0,0.1,25.3,25.3,80.0,0.004,0.0205,0.0595,NaN
2,2.0,1.9,25.3,25.4,86.1,0.094,0.1907,0.8030,248.8
3,2.0,2.9,25.3,25.4,86.1,0.146,0.6398,2.2931,172.0
4,2.0,3.1,25.3,25.5,75.2,0.157,1.1065,3.6281,28.3
...,...,...,...,...,...,...,...,...,...
765,1.0,2.7,25.3,25.5,85.0,0.134,0.6289,1.8671,125.1
766,1.0,2.1,25.3,25.5,85.0,0.106,0.5446,2.2778,53.8
767,1.0,1.3,25.3,25.5,85.0,0.066,0.2120,1.9588,316.9
768,1.0,2.3,25.3,25.5,85.0,0.116,0.3053,1.2578,287.1


In [38]:
pca = learn(healthy)

In [39]:
testdata = load_data("test_data.csv")
report = test(pca, testdata)
report

,disagreement,is_unusual,suspect_column,drift_estimate,corrected_value,note
0,NaN,False,NaN,NaN,NaN,has empty cells - not scored
1,NaN,False,NaN,NaN,NaN,has empty cells - not scored
2,NaN,False,NaN,NaN,NaN,has empty cells - not scored
3,0.183675,False,NaN,NaN,NaN,
4,0.428742,False,NaN,NaN,NaN,
...,...,...,...,...,...,...
295,6.935709,True,ambient_temp_C,-1.269426,26.556726,
296,6.69643,True,ambient_temp_C,-1.223692,26.508092,
297,7.437305,True,ambient_temp_C,-1.227942,26.552342,
298,8.284816,True,ambient_temp_C,-1.292917,26.600117,


In [40]:
truth = pd.read_csv("test_truth.csv")

In [41]:
summary, errors = compare(report, truth)

KeyError: 'true_load_percent_drift'